# Data Ingestion – Bronze Layer

This notebook performs a **full load** data ingestion from the **raw layer (Volume)** to the **bronze layer**, using **CSV** and **Excel** files and persisting the data in **Delta Lake** format.

### Parameterization
The process is parameterized through Databricks widgets:
- `catalog`: Target catalog  
- `schema`: Target schema  
- `table`: Dataset identifier, used both in the **input path (raw)** and in the **target table (bronze)**  


In [0]:
import pandas as pd
from pyspark.sql.functions import current_timestamp, col

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table")

# Base path inside the Volume
path = f"/Volumes/raw/take_home_test/full_load/{table}/"

In [0]:
# List all files in the directory
files_raw = spark.sql(f"LIST '{path}'")

# Extract file paths
files = [row.path for row in files_raw.collect()]

dfs = []

In [0]:
for fpath in files:
    format_file = fpath.lower().split(".")[-1]
    print(f"Processing: {fpath}")

    # Read CSV files using Spark
    if format_file == "csv":
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("sep", ",")
            .load(fpath)
        )

    # Read Excel files using pandas
    elif format_file in ["xlsx", "xls"]:
        # Direct read from Volume
        pdf = pd.read_excel(fpath)

        # Force all columns to string (prevents Arrow errors)
        pdf = pdf.astype(str)

        # Convert pandas to spark
        df = spark.createDataFrame(pdf)

    else:
        print(f"Skipping unsupported file: {fpath}")
        continue

    dfs.append(df)

In [0]:
# Union all files with schema alignment
df_full = dfs[0]
for d in dfs[1:]:
    df_full = df_full.unionByName(d, allowMissingColumns=True)

# Add metadata column
df_full = df_full.withColumn("inserted_at", current_timestamp())

In [0]:
(df_full.coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.{table}"))